In [3]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb
from merge_tables.db.tables import create_clean_account_name_macro

from pathlib import Path


MERGE_TABLES_DIR = Path('.').parent 
DATA_DIR = MERGE_TABLES_DIR / "data"
OUTPUT_DIR = MERGE_TABLES_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
duck = connect_to_postgres_via_duckdb()
create_clean_account_name_macro(duck)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'
✓ Created clean_account_name macro


# Merger process


## 1. Get all medisoft frims

In [5]:
duck.sql(
    """
    create or replace table medisoft_firms as 
    select *
    from pg.medisoft.table_firmenstruktur
    """
)

## 1.1 Get all easybill customers

In [6]:
duck.sql(
    """
    create or replace table easybill_customers as 
    select
        distinct on ("Kontakt: Kundennummer") 
        *
    from pg.easybill.contacts
    """
)

In [7]:
duck.sql("select * from medisoft_firms")

┌──────────────────────────────────────┬────────────────────────────────────────┬────────────────────────────────────────┬─────────────────────────────────────────────────────┬─────────┬────────────────┬───────────────┬───────────────┬─────────────────┬───────────────┬───────────────┬───────────────┬────────────────┬──────────────────────────────────────────┬────────────┬─────────────────┬──────────┬─────────┬───────────┬────────────┬─────────────────┬────────────────────────────────┬─────────┬──────────────────────┬────────────┬───────────────┬─────────┬───────────────┬────────────────────────────────────────────┬──────────────────┬────────────────┬─────────────────────────┬─────────┬──────────┬─────────┬───────────────────┬────────────┬─────────────────────────┬────────────┬─────────────┬─────────────┬────────────────────┐
│                rec_id                │                kuerzel                 │                  name                  │                        pfad             

In [8]:
duck.sql("select * from easybill_customers")

┌─────────────────────┬───────────────────────┬────────────────────────────┬────────────────────┬─────────────────┬─────────────────────────────────┬────────────────┬──────────────────┬───────────────┬───────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────┬───────────────────────┬───────────────────┬─────────────────────┬───────────────┬──────────────────────┬───────────────────┬────────────────────────────────┬───────────────────────┬──────────────────────────────┬────────────────────────┬───────────────────────────────┬──────────────────────┬────────────────────┬──────────────┬───────────────────────┬─────────────────────────────────┬───────────────────────────┬───────────────────┬─────────────────┬─────────────────┬─────────────────────┬──────────────────────────────┬──────────────────────────┬─────────────────────────┬───────────────────────┬───────────────────┬──────────────────────┬─

## 2.1 Find the medisoft customers that match with the medisoft firms

In [9]:
duck.sql(
    """
    create or replace table broad_match as 
    with medisoft_data as (
        select
            rec_id as medisoft_id,
            case 
                when trim(split(pfad, '/')[2]) = 'Nicht Kunden' or trim(split(pfad, '/')[2]) = 'Kunden' then clean_account_name(split(pfad, '/')[3])
                else clean_account_name(split(pfad, '/')[2])
            end as mother_entity_name, 
            name, 
            kuerzel,
            pfad,
            split(pfad, '/') as s,
            strasse,
            plz,
            ort,
            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from medisoft_firms
    ), easybill_data as (
        select
            *,
            clean_account_name(coalesce("Kontakt: Firma", concat_ws(' ', "Kontakt: Vorname", "Kontakt: Name"))) as clean_firmenname,
        from easybill_customers
    )
    select
        *,
        greatest(
            jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_data.clean_firmenname),
            jaro_winkler_similarity(medisoft_data.clean_name, easybill_data.clean_firmenname),
            jaro_winkler_similarity(medisoft_data.clean_kuerzel, easybill_data.clean_firmenname)
        ) as sim
    from medisoft_data
    left join easybill_data
        on jaro_winkler_similarity(medisoft_data.mother_entity_name, easybill_data.clean_firmenname) > 0.95
        or jaro_winkler_similarity(medisoft_data.clean_name, easybill_data.clean_firmenname) > 0.95
        or jaro_winkler_similarity(medisoft_data.clean_kuerzel, easybill_data.clean_firmenname) > 0.95
    order by name
    """
)

## 3. Isolate results

### 3.1 validated matchs

In [12]:
duck.sql(
    """
    select * from broad_match where medisoft_id in (
        select medisoft_id from broad_match where  "Kontakt: Kundennummer" is not null
        group by medisoft_id
        having count(*) = 1
    )
    """
)

┌───────────────┬───────────────────────────────┬──────────────────────────────────────────────────────┬─────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────┬────────────────────────────┬─────────┬───────────────────┬───────────────────────────────┬───────────────────────────────┬─────────────────────┬───────────────────────┬────────────────────────────┬────────────────────┬─────────────────┬─────────────────────────────────┬────────────────┬──────────────────┬───────────────┬───────────────────────┬────────────────────────────────────────────────┬────────────────────────────┬───────────────────────┬───────────────────┬─────────────────────┬───────────────┬──────────────────────┬───────────────────┬────────────────────────────────┬───────────────────────┬──────────────────────────────┬────────────────────────┬───────────────────────────────┬────────

In [40]:
duck.sql(
    """
    with found as (
            select * from broad_match where medisoft_id in (
            select medisoft_id from broad_match where  "Kontakt: Kundennummer" is not null
            group by medisoft_id
            having count(*) = 1
        )
    )
    select 
        medisoft_id,
        coalesce(name, kuerzel) as name,
        pfad,
        "Kontakt: Kundennummer" as easybill_kundennummer,
        "Kontakt: Firma" as easybill_firma,
        "Kontakt: Ort" as easybill_ort,
        sim,
        case
            when sim < 1 then false
            else true
        end as validated
    from found 
    """
).to_csv("output/validated_matches.csv")

### 3.2 Isolate medisoft firms with multiple easybill matches

Keep only rows where one medisoft firm (medisoft_id) matched more than one easybill customer.

In [9]:
duck.sql(
    """
    with multi_match_firms as (
        select medisoft_id, count(*) as c
        from broad_match
        where "Kontakt: Kundennummer" is not null
        group by medisoft_id
        having count(*) > 1
    )
    select medisoft_id, name, pfad, c  from multi_match_firms join medisoft_firms on medisoft_id = rec_id
    order by c asc, name
    """
).show(max_rows=10500)

┌───────────────┬────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┐
│  medisoft_id  │                                        name                                        │                                                        pfad                                                         │   c   │
│    varchar    │                                      varchar                                       │                                                       varchar                                                       │ int64 │
├───────────────┼────────────────────────────────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼───────┤
│ 00_8Z600VXDWJ │  FabFab GmbH                                                      

#### 3.2.1 for all 1-to-4 max matches, write into a spreadsheet to manually validate choices

In [21]:
# 1-to-4 matches: write to Google Sheet for manual validation (match_is_good = True/False)
# Setup: pip install gspread pandas; create a Google Cloud service account with Sheets API,
# download JSON key, share the spreadsheet with the service account email as Editor,
# set GOOGLE_APPLICATION_CREDENTIALS to the JSON path or pass credentials_path below.

import os
import pandas as pd

SPREADSHEET_ID = "1Fclu_yVR-pv7y_EQ-tEwk1RGValdIPZ8QE7M3Ep_AeY"
SHEET_NAME = "Sheet1"  # or your tab name (gid=0 is first sheet)
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

df = duck.sql(
    """
    with multi_match_firms as (
        select medisoft_id, count(*) as c
        from broad_match
        where "Kontakt: Kundennummer" is not null
        group by medisoft_id
        having count(*) between 2 and 5
    )
    select
        broad_match.medisoft_id,
        broad_match.name as medisoft_name,
        broad_match.pfad as medisoft_pfad,
        broad_match."Kontakt: Kundennummer" as easybill_kundennummer,
        broad_match."Kontakt: Firma" as easybill_firma,
        broad_match."Kontakt: Ort" as easybill_ort,
        broad_match.clean_firmenname,
        round(broad_match.sim::double, 4) as similarity,
        false as match_is_good
    from broad_match
    join multi_match_firms using (medisoft_id)
    where "Kontakt: Kundennummer" is not null
    order by medisoft_id, similarity desc
    """
).df()

# Write to Google Sheet (requires gspread + auth)
try:
    import gspread
    from google.oauth2.service_account import Credentials

    scopes = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
    creds = Credentials.from_service_account_file(credentials_path, scopes=scopes) if credentials_path else None
    if not creds:
        raise FileNotFoundError("Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path")

    gc = gspread.authorize(creds)
    sh = gc.open_by_key(SPREADSHEET_ID)
    worksheet = sh.worksheet(SHEET_NAME)

    # Prepare header + data; match_is_good as empty so you can type TRUE/FALSE in the sheet
    df_export = df.copy()
    df_export["match_is_good"] = ""  # leave empty for manual True/False
    values = [df_export.columns.tolist()] + df_export.fillna("").astype(str).values.tolist()

    worksheet.clear()
    worksheet.update(values, value_input_option="USER_ENTERED")
    print(f"✓ Written {len(df_export)} rows to spreadsheet sheet '{SHEET_NAME}'")
except Exception as e:
    print("Google Sheet write skipped (install gspread, google-auth; set GOOGLE_APPLICATION_CREDENTIALS):", e)
    df.to_csv(OUTPUT_DIR / "multi_match_1_to_4_validation.csv", index=False)
    print("✓ Fallback: saved to", OUTPUT_DIR / "multi_match_1_to_4_validation.csv")


Google Sheet write skipped (install gspread, google-auth; set GOOGLE_APPLICATION_CREDENTIALS): No module named 'gspread'
✓ Fallback: saved to output/multi_match_1_to_4_validation.csv


In [20]:
duck.sql(
    """
        with multi_match_firms as (
        select medisoft_id, count(*) as c
        from broad_match
        where "Kontakt: Kundennummer" is not null
        group by medisoft_id
        having count(*) > 5
    )
    select mother_entity_name, count(*) as c from broad_match where medisoft_id in (select medisoft_id from multi_match_firms) group by mother_entity_name order by c desc
    """
)

┌────────────────────────────────────────┬───────┐
│           mother_entity_name           │   c   │
│                varchar                 │ int64 │
├────────────────────────────────────────┼───────┤
│ ejf                                    │  8067 │
│ lebenshilfewerkkreisherzogtumlauenburg │   194 │
│ lebenshilfewerkhagenow                 │   180 │
│ medicover                              │   126 │
│ fairdoctors                            │    56 │
│ lebenshilfewerkhagenowwfb              │    36 │
│ schneiderschere                        │    24 │
│ medicovermittemvz                      │    13 │
│ medicovermohrenstr                     │    11 │
│ aufdertenneevloschen                   │    11 │
│ bundesanstalttechnischeshilfswerk      │     6 │
│ schneiderscherebfi                     │     6 │
│ schneiderscherepff                     │     6 │
├────────────────────────────────────────┴───────┤
│ 13 rows                              2 columns │
└──────────────────────────────

#### 3.2.2 5+ matches: one Excel file per mother_entity_name

Each file has 3 sheets: **Medisoft** (firms), **Easybill** (match options), **Choose** (medisoft info + dropdown to pick the right easybill).

In [28]:
# 5+ matches: one Excel per mother_entity_name (Sheet1=Medisoft, Sheet2=Easybill, Sheet3=Choose + dropdown)
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation

# Mother entities with 5+ easybill matches (by medisoft_id)
mothers_5plus = duck.sql("""
    with multi_match_firms as (
        select medisoft_id, count(*) as c
        from broad_match
        where "Kontakt: Kundennummer" is not null
        group by medisoft_id
        having count(*) > 5
    )
    select distinct mother_entity_name
    from broad_match
    where medisoft_id in (select medisoft_id from multi_match_firms)
    order by mother_entity_name
""").df()

def safe_filename(name):
    return re.sub(r'[^\w\-_]', '_', (name or "unnamed")[:80]).strip("_") or "unnamed"

for _, row in mothers_5plus.iterrows():
    mother = row["mother_entity_name"]
    key = safe_filename(str(mother))
    mother_esc = str(mother).replace("'", "''")  # SQL escape

    medisoft_df = duck.sql(f"""
        select distinct medisoft_id, mother_entity_name, name, kuerzel, pfad
        from broad_match
        where mother_entity_name = '{mother_esc}'
        and "Kontakt: Kundennummer" is not null
    """).df()

    easybill_df = duck.sql(f"""
        select distinct
            "Kontakt: Kundennummer" as easybill_kundennummer,
            "Kontakt: Firma" as easybill_firma,
            "Kontakt: Ort" as easybill_ort,
            clean_firmenname,
            round(sim::double, 4) as similarity
        from broad_match
        where mother_entity_name = '{mother_esc}'
        and "Kontakt: Kundennummer" is not null
        order by similarity desc
    """).df()

    easybill_df["choice_label"] = (
        easybill_df["easybill_kundennummer"].astype(str)
        + " | " + (easybill_df["easybill_firma"].fillna("")).astype(str)
        + " | " + (easybill_df["easybill_ort"].fillna("")).astype(str)
    )

    out_path = OUTPUT_DIR / f"5plus_{key}.xlsx"
    with pd.ExcelWriter(out_path, engine="openpyxl") as wb:
        medisoft_df.to_excel(wb, sheet_name="Medisoft", index=False)
        easybill_df.to_excel(wb, sheet_name="Easybill", index=False)
        # Sheet3: all medisoft firms + dropdown to pick easybill for each
        choose = medisoft_df.copy()
        choose["selected_easybill"] = ""
        choose.to_excel(wb, sheet_name="Choose", index=False)

    # Add dropdown on Sheet "Choose" for selected_easybill (list from Easybill choice_label)
    book = load_workbook(out_path)
    ws_choose = book["Choose"]
    ws_easy = book["Easybill"]
    n_easy = len(easybill_df) + 1
    list_range = f"'Easybill'!$F$2:$F${n_easy}"
    dv = DataValidation(type="list", formula1=list_range, allow_blank=True)
    dv.error = "Pick an option from the list"
    ws_choose.add_data_validation(dv)
    end_row = max(2, len(medisoft_df) + 1)
    dv.add(f"F2:F{end_row}")
    book.save(out_path)
    print(f"  {out_path.name}")

print(f"✓ Written {len(mothers_5plus)} Excel files to {OUTPUT_DIR}")

  5plus_aufdertenneevloschen.xlsx
  5plus_bundesanstalttechnischeshilfswerk.xlsx
  5plus_ejf.xlsx
  5plus_fairdoctors.xlsx
  5plus_lebenshilfewerkhagenow.xlsx
  5plus_lebenshilfewerkhagenowwfb.xlsx
  5plus_lebenshilfewerkkreisherzogtumlauenburg.xlsx
  5plus_medicover.xlsx
  5plus_medicovermittemvz.xlsx
  5plus_medicovermohrenstr.xlsx
  5plus_schneiderschere.xlsx
  5plus_schneiderscherebfi.xlsx
  5plus_schneiderscherepff.xlsx
✓ Written 13 Excel files to output


In [10]:
# Medisoft firms that have multiple easybill matches (one-to-many)
duck.sql(
    """
    with multi_match_firms as (
        select medisoft_id, count(*) as c
        from broad_match
        where "Kontakt: Kundennummer" is not null
        group by medisoft_id
        having count(*) > 1
    )
    select broad_match.medisoft_id, broad_match.name, broad_match."Kontakt: Kundennummer", broad_match."Kontakt: Firma", broad_match.clean_firmenname, c
    from broad_match
    join multi_match_firms using (medisoft_id)
    where c <10
    order by c asc, medisoft_id
    """
).show(max_rows=10500)

┌───────────────┬────────────────────────────────────────────────────────────────────────────┬───────────────────────┬────────────────────────────────────────────────────────────┬───────────────────────────────────────────────┬───────┐
│  medisoft_id  │                                    name                                    │ Kontakt: Kundennummer │                       Kontakt: Firma                       │               clean_firmenname                │   c   │
│    varchar    │                                  varchar                                   │        varchar        │                          varchar                           │                    varchar                    │ int64 │
├───────────────┼────────────────────────────────────────────────────────────────────────────┼───────────────────────┼────────────────────────────────────────────────────────────┼───────────────────────────────────────────────┼───────┤
│ 00_8HW00JGCSQ │ la Red GmbH                           

### 3.3 Isolate medisoft firms with no easybill match

In [23]:
duck.sql(
    """
    select 
        distinct on(medisoft_id)
        medisoft_id,
        name,
        kuerzel,
        pfad,
        m.ort,
        m.plz,
        m.strasse,
        count(distinct b.rec_id)
        
    from broad_match m
    left join pg.medisoft.table_beschaeftigte as b
        on m.medisoft_id = b.ebetrieb_id or m.medisoft_id = b.abetrieb_id
    where "Kontakt: Kundennummer" is null 
    group by all
    order by pfad
    """
)
#.to_csv("output/no_easybill_match.csv")

┌──────────────────────────────────────┬──────────────────────────────────────────────────┬─────────────────────────────────────────┬──────────────────────────────────────────────────────────────────┬────────────────────┬─────────┬─────────────────────────────┬──────────────────────────┐
│             medisoft_id              │                       name                       │                 kuerzel                 │                               pfad                               │        ort         │   plz   │           strasse           │ count(DISTINCT b.rec_id) │
│               varchar                │                     varchar                      │                 varchar                 │                             varchar                              │      varchar       │ varchar │           varchar           │          int64           │
├──────────────────────────────────────┼──────────────────────────────────────────────────┼─────────────────────────────────────────┼

### 3.4 Count how many easybill are used over the total

In [12]:
duck.sql(
    """
    select count(distinct "Kontakt: Kundennummer") from broad_match union all select count(distinct "Kontakt: Kundennummer") from easybill_customers where "Kontakt: Firma" is not null
    """
)

┌─────────────────────────────────────────┐
│ count(DISTINCT "Kontakt: Kundennummer") │
│                  int64                  │
├─────────────────────────────────────────┤
│                                    1437 │
│                                    2641 │
└─────────────────────────────────────────┘